# TRELLIS Colab preflight only

No installs, weight downloads, generation, web server or Drive mount. Running a paid GPU runtime may consume compute units. Use only after the owner authorizes a bounded trial. Read README.md first.

This notebook has been syntax-checked locally, not run in Colab. Hardware detection does not prove software compatibility. The 22,000 MiB diagnostic boundary only rejects clearly undersized GPUs; it is not a revised model minimum.

In [ ]:
import json, platform, shutil, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone

def command(args):
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=30)
        return {"returncode": p.returncode, "stdout": p.stdout.strip(), "stderr": p.stderr.strip()}
    except (OSError, subprocess.TimeoutExpired) as exc:
        return {"error": type(exc).__name__}

report = {
    "checked_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Hardware/software inventory only; no model run or installation",
    "python": sys.version.split()[0],
    "system": platform.system(),
    "gpu": command(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]),
    "cuda_compiler": command(["nvcc", "--version"]),
    "disk_free_gib": round(shutil.disk_usage(Path.cwd()).free / 2**30, 1),
    "upstream_revision": "75fbf0183001ed9876c8dbb35de6b68552ee08bd",
}
try:
    import torch
    report["torch"] = {"version": torch.__version__, "cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available()}
except ImportError:
    report["torch"] = {"installed": False}
print(json.dumps(report, indent=2))


In [ ]:
gpu_text = report["gpu"].get("stdout", "")
messages = []
for line in gpu_text.splitlines():
    fields = [x.strip() for x in line.split(",")]
    if len(fields) != 3:
        continue
    name, memory_mib, _ = fields
    try:
        memory_mib = float(memory_mib)
    except ValueError:
        continue
    if memory_mib < 22000:
        verdict = "Below the documented 24 GB class requirement. Do not start TRELLIS.2."
    elif "A100" in name or "H100" in name:
        verdict = "Upstream-verified GPU family; local software/install/inference are still untested."
    else:
        verdict = "Possible hardware candidate only. Verify advertised VRAM and extension compatibility; not upstream-verified here."
    messages.append({"gpu": name, "memory_mib": memory_mib, "verdict": verdict})
report["assessment"] = messages or [{"verdict": "No usable NVIDIA GPU inventory. Do not start inference."}]
print(json.dumps(report["assessment"], indent=2))
# Write a new receipt without overwriting any prior file.
receipt = Path("trellis_preflight_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + ".json")
with receipt.open("x", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)
print("Receipt:", receipt.name)
